# 2-4 データの加工

前回までで「読み込み」「集計」「絞り込み」ができるようになりました。今回は、データに **手を加える** 操作を扱います。

実務で言う **「ヘルパー列を追加」「日付から月だけ取り出す」「空欄を埋める」** といった処理です。

## このノートのゴール

- 既存の列から計算して **新しい列を追加** できる
- 列の **リネーム・削除** ができる
- `.dt` で日付から **年・月・曜日** を取り出せる
- `.str` で文字列を **検索・置換** できる
- **欠損値 (NaN)** を発見し、埋める／除外できる
- `apply()` で **自分で書いた関数** を全行に適用できる（1-5 と 2-3 の橋渡し）

## Excel / VBA との対応表

| やりたいこと | Excel / VBA | pandas |
|---|---|---|
| ヘルパー列を作る | `=A2*0.1` を新しい列にドラッグ | `df["新列"] = df["A"] * 0.1` |
| 列の見出しを変える | セルの値を編集 | `df.rename(columns={...})` |
| 列を消す | 列を選択して削除 | `df.drop(columns=[...])` |
| 日付から月を取り出す | `=MONTH(A2)` | `df["date"].dt.month` |
| 空欄を埋める | `IF(ISBLANK(A2), "-", A2)` | `df["列"].fillna("-")` |
| 文字列を置換 | `SUBSTITUTE(A2, "旧", "新")` | `df["列"].str.replace("旧", "新")` |
| 自作の関数を全行に | `Function` を VBA で定義してセルから呼ぶ | `df["列"].apply(自作関数)` |

## データの準備

2-3 と同じ売上データを使います。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
BASE = "/content/drive/MyDrive/python-data-basics"
DATA = f"{BASE}/data/pandas"

df = pd.read_excel(f"{DATA}/sales_2026-01.xlsx", sheet_name="売上", parse_dates=["date"])
df.head()

## 1. 新しい列を追加する

**Excel でヘルパー列を作るのと同じ感覚です。**

書き方:
```python
df["新しい列名"] = 計算式
```

計算式には、既存の列を使えます。

In [ ]:
# (A) 単純な計算: 税額 (10%) の列を追加
df["tax"] = df["amount"] * 0.1

# (B) 複数列を組み合わせる: 税込金額
df["amount_with_tax"] = df["amount"] + df["tax"]

# (C) 文字列の組み立て: 商品名 × 数量 のラベル
#     数値は astype(str) で文字列に変換してから連結
df["label"] = df["product_name"] + " × " + df["quantity"].astype(str)

df.head()

## 2. 列のリネーム・削除

Excel で「列見出しを書き換える」「列を消す」のと同じ操作です。

| 操作 | 書き方 |
|---|---|
| 列名を変える | `df = df.rename(columns={"旧": "新"})` |
| 列を消す | `df = df.drop(columns=["列1", "列2"])` |

**ポイント**: どちらも **新しい DataFrame を返す** ので、`df = ...` で代入し直します。

In [ ]:
# 列名を日本語に変える（レポート出力に使うとき便利）
df = df.rename(columns={
    "date": "日付",
    "product_name": "商品名",
    "amount": "売上",
})

# tax 列はもう要らないので削除
df = df.drop(columns=["tax", "amount_with_tax", "label"])

df.head()

## 3. 日付の処理 (`.dt` アクセサ)

日付型の列には `.dt` を付けると、**年・月・日・曜日** などを取り出せます。

Excel の `YEAR()` `MONTH()` `WEEKDAY()` に相当します。

| コード | 取り出すもの | Excel |
|---|---|---|
| `.dt.year` | 年 | `YEAR()` |
| `.dt.month` | 月 | `MONTH()` |
| `.dt.day` | 日 | `DAY()` |
| `.dt.dayofweek` | 曜日番号（**月=0, 日=6**） | `WEEKDAY()` |

In [ ]:
# 年・月・日を取り出す
df["年"] = df["日付"].dt.year
df["月"] = df["日付"].dt.month
df["日"] = df["日付"].dt.day

# 曜日: dayofweek は 0=月 … 6=日 という数値なので、辞書で日本語に変換
#       (1-3 で習った dict + Series.map の組み合わせ)
day_jp = {0: "月", 1: "火", 2: "水", 3: "木", 4: "金", 5: "土", 6: "日"}
df["曜日"] = df["日付"].dt.dayofweek.map(day_jp)

df[["日付", "年", "月", "日", "曜日"]].head()

## 4. 文字列の処理 (`.str` アクセサ)

文字列の列には `.str` を付けると、**検索・置換・分割** などができます。

| コード | 意味 | Excel |
|---|---|---|
| `.str.contains("X")` | X を含むか (True/False) | `ISNUMBER(SEARCH(...))` |
| `.str.replace("旧", "新")` | 置換 | `SUBSTITUTE` |
| `.str.startswith("X")` | X で始まるか | `LEFT(...) = "X"` |
| `.str.len()` | 文字数 | `LEN` |

In [ ]:
# (A) 商品名に「パック」を含むか
df["is_pack"] = df["商品名"].str.contains("パック")

# (B) 商品名から末尾の " (...)" を消す
#     例: 「りんご (1袋)」→「りんご」
df["短縮名"] = df["商品名"].str.split(" (").str[0]

df[["商品名", "is_pack", "短縮名"]].head(8)

## 5. 欠損値 (`NaN`) の扱い

実務で Excel を読み込むと、**「空欄のセル」** がよくあります。pandas ではこれが **`NaN`** （Not a Number、欠損値）として現れます。

ここでは欠損ありのデータを自分で作って、扱い方を見ます。

| やりたいこと | コード | Excel |
|---|---|---|
| 欠損があるかチェック | `df.isna()` / `df.isna().sum()` | `ISBLANK` |
| 欠損を埋める | `df["列"].fillna(値)` | `IFERROR` / `IF(ISBLANK(...))` |
| 欠損のある行を消す | `df.dropna()` | フィルタで空白を除外 |

In [ ]:
# 欠損のあるサンプルデータを作る（実際の Excel 取り込みでよく見る形）
sample = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "name": ["田中", "鈴木", None, "山田", "佐藤"],
    "salary": [350000, 420000, 380000, None, 320000],
})
print("=== 元のデータ (None / NaN が欠損) ===")
print(sample)

print("\n=== 各列の欠損件数 ===")
print(sample.isna().sum())

print("\n=== 欠損を埋める (name は '(不明)'、salary は 0) ===")
print(sample.fillna({"name": "(不明)", "salary": 0}))

print("\n=== 欠損のある行を消す ===")
print(sample.dropna())

## 6. `apply()` で関数を全行に適用

1-5 で書いた **自作の関数** を、列のすべての値に一気に適用できます。

Excel/VBA で言えば、`Function` で自作関数を定義して、それをセルから呼び出すのと同じ感覚です。

書き方:
```python
df["新列"] = df["元の列"].apply(関数名)
```

In [ ]:
# 1-5 で書いた「ランク分け」を関数として定義
def classify(amount):
    if amount >= 20000:
        return "大口"
    elif amount >= 10000:
        return "中口"
    else:
        return "小口"

# 全行に適用して新しい列に
df["ランク"] = df["売上"].apply(classify)

df[["日付", "商品名", "売上", "ランク"]].head(10)

## 練習問題

1. **単価が 500 円以上** かどうかを示す `is_premium` 列（True/False）を追加してください
2. **「○月○日」という文字列** の列 `表示日付` を作ってください（例: 「1月15日」）
3. **数量** から、5 個以上を `"大量"`、3〜4 個を `"中"`、それ以下を `"少"` と返す関数 `size_label` を書いて、`order_size` 列を追加してください
4. **`曜日` が「土」または「日」** の行だけを取り出してください（2-3 の絞り込みの復習）

In [ ]:
# ここに書いてみてください



<details>
<summary>答えを見る</summary>

```python
# 1. is_premium 列
df["is_premium"] = df["unit_price"] >= 500

# 2. 表示日付
df["表示日付"] = df["月"].astype(str) + "月" + df["日"].astype(str) + "日"
# ↑ 2 と 3 が「2月3日」と並ぶことを許容するなら上で OK。
#   .dt.strftime で書く方法もあり: df["日付"].dt.strftime("%-m月%-d日")

# 3. order_size
def size_label(qty):
    if qty >= 5:
        return "大量"
    elif qty >= 3:
        return "中"
    else:
        return "少"
df["order_size"] = df["quantity"].apply(size_label)

# 4. 土日だけ
df[df["曜日"].isin(["土", "日"])]
```

</details>

## よくあるエラー

### 1. `rename` / `drop` の結果を `df = ...` に代入し忘れる
```python
df.rename(columns={"a": "A"})  # ❌ 結果が消えてしまう
```
→ **`df = df.rename(...)`** と代入する。

### 2. `.dt` を文字列の列に使ってしまう
```python
df["date"].dt.year  # ❌ date が文字列だと AttributeError
```
→ `pd.read_excel(..., parse_dates=["date"])` で読み込み時に日付型にする。途中で変換するなら `pd.to_datetime(df["date"])`。

### 3. 文字列と数値を `+` で連結
```python
df["商品名"] + df["quantity"]  # ❌ TypeError
```
→ 数値は `.astype(str)` で文字列に変換してから連結する。

### 4. `apply` に渡す関数の **呼び出し** を書いてしまう
```python
df["売上"].apply(classify(0))  # ❌ classify(0) は結果の文字列を渡している
```
→ **`(...)` を付けない関数名そのもの** を渡す: `df["売上"].apply(classify)`

## まとめ

- `df["新列"] = ...` で列を **追加**、`rename` / `drop` で **見出し変更・削除**
- `.dt` で **日付**、`.str` で **文字列** の処理
- `isna` / `fillna` / `dropna` で **欠損値** に対応
- `apply(関数)` で **自作関数を全行に**

次は **2-5「Excel への書き出し」** で、加工した DataFrame を Excel ファイルとして保存する方法を扱います。